# 01 — Exploratory Data Analysis

**Scope.** Full CAMS reanalysis history for both forecast zones — `capital`
(Islamabad + Rawalpindi, one grid cell, ADR-008) and `lahore` — read straight
from the feature store (`data/feature_store/`, D3/D4). This notebook answers
one question per section and states the finding in the text immediately
after the chart that produced it; no chart appears without a sentence saying
what it shows and why it matters (CLAUDE.md, session 4 brief).

**What "AQI" means here.** All AQI values in this notebook are our own
computation (`hourly_aqi_nowcast`, EPA NowCast over PM2.5, 2024 breakpoints —
`aqi_scale.py`), from **CAMS reanalysis**, not station measurement (ADR-001).
The model-vs-station gap is a separate, dedicated question —
`02_divergence.ipynb`.

**Reproducibility.** This notebook imports only from `src/aqi` (CLAUDE.md
§16) and reads whatever is currently in the feature store — re-running it
after another backfill or hourly pipeline run will reflect the larger
window, not this session's numbers verbatim. Every figure is also saved to
`reports/figures/` so the report and D11's evidence do not depend on the
notebook's own (git-ignored, per §16) outputs.


In [ ]:
import matplotlib
matplotlib.use("Agg")  # headless — this notebook is executed by CI/CLI, not a live kernel

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from aqi.config import get_config
from aqi.store.parquet_store import ParquetFeatureStore

FIGURES = Path("../reports/figures")
FIGURES.mkdir(parents=True, exist_ok=True)

ZONE_LABELS = {"capital": "Islamabad/Rawalpindi", "lahore": "Lahore"}
ZONE_COLORS = {"capital": "#1f77b4", "lahore": "#d62728"}
SMOG_MONTHS = {10, 11, 12, 1, 2}  # CLAUDE.md §9, conf/config.yaml test_window_must_include_months

store = ParquetFeatureStore()
config = get_config()
group = config.store.feature_group

df = store.read(group, pd.Timestamp("2000-01-01", tz="UTC"), pd.Timestamp.now(tz="UTC"))
df["time_utc"] = pd.to_datetime(df["time_utc"], utc=True)
df["local_time"] = df["time_utc"].dt.tz_convert("Asia/Karachi")
df["local_date"] = df["local_time"].dt.date
df["local_month"] = df["local_time"].dt.month
df["local_hour"] = df["local_time"].dt.hour
df["local_year"] = df["local_time"].dt.year
df["smog_season"] = df["local_month"].isin(SMOG_MONTHS)

print(f"{len(df):,} rows, {df['city_id'].nunique()} zones")
for zone_id, zone_df in df.groupby("city_id"):
    span = zone_df["local_time"].min(), zone_df["local_time"].max()
    print(f"  {zone_id:10s} {len(zone_df):,} rows  {span[0].date()} .. {span[1].date()}")


## 1. The smog-season signature

Punjab's winter smog is the reason this project exists (CLAUDE.md §1). The
first question EDA has to answer is whether the feature store actually shows
it — if the seasonal signal isn't there in four years of reanalysis, nothing
built on top of it (physics features, episode metrics, alerts) has a signal
to work with.


In [ ]:
monthly = (
    df.groupby(["city_id", "local_month"])
    .agg(pm2_5=("pm2_5", "mean"), aqi=("hourly_aqi_nowcast", "mean"))
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharex=True)
months = range(1, 13)
month_labels = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

for zone_id in ["capital", "lahore"]:
    sub = monthly[monthly.city_id == zone_id].set_index("local_month").reindex(months)
    axes[0].plot(months, sub["pm2_5"], marker="o", label=ZONE_LABELS[zone_id], color=ZONE_COLORS[zone_id])
    axes[1].plot(months, sub["aqi"], marker="o", label=ZONE_LABELS[zone_id], color=ZONE_COLORS[zone_id])

for ax, title, ylabel in zip(
    axes, ["Mean PM2.5 by month", "Mean hourly AQI (NowCast) by month"], ["µg/m³", "AQI"]
):
    ax.axvspan(0.5, 2.5, color="grey", alpha=0.12)
    ax.axvspan(9.5, 12.5, color="grey", alpha=0.12, label="_nolegend_")
    ax.set_title(title)
    ax.set_xticks(months)
    ax.set_xticklabels(month_labels)
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.25)

axes[0].legend(loc="upper center")
fig.suptitle("Monthly climatology, both zones, full backfill window (shaded = Oct–Feb smog season)")
fig.tight_layout()
fig.savefig(FIGURES / "eda_monthly_climatology.png", dpi=140)
plt.show()

for zone_id in ["capital", "lahore"]:
    sub = monthly[monthly.city_id == zone_id].set_index("local_month")
    worst_m, best_m = sub["aqi"].idxmax(), sub["aqi"].idxmin()
    print(
        f"{zone_id:10s} worst month {month_labels[worst_m-1]} (AQI {sub['aqi'][worst_m]:.0f}), "
        f"best month {month_labels[best_m-1]} (AQI {sub['aqi'][best_m]:.0f}), "
        f"ratio {sub['aqi'][worst_m] / sub['aqi'][best_m]:.1f}x"
    )


**Finding.** Both zones peak in January and trough in April, and the shaded
Oct–Feb window in the chart above visibly contains the entire climb and the
start of the descent — the smog season definition CLAUDE.md §9.1/I2 uses for
the held-out test window lines up with where the data actually is worst, not
an assumption. Lahore's amplitude is far larger than the capital's: its worst
month runs roughly 2x its best month's AQI, while the capital's ratio is
milder — consistent with Lahore sitting inside the Indo-Gangetic basin
(shared airshed with Indian Punjab crop burning, denser traffic and
industry) while Islamabad/Rawalpindi sit on the Potohar plateau, partly
outside it (ADR-007's own finding about the two cities' airsheds). This is
the first quantitative support for modelling the two zones separately
(ADR-008/ADR-013) rather than pooling them — they don't just have different
coordinates, they have different seasonal amplitude.


## 2. Diurnal structure

Does PM2.5 have a within-day pattern, and does that pattern change in smog
season? A flat diurnal profile would suggest the target is driven almost
entirely by synoptic/day-to-day weather; a strong one says time-of-day
features (already in the feature store as `hour_sin`/`hour_cos`) are
pulling real weight, and it separately tells us *when* an alert is most
likely to be triggered.


In [ ]:
diurnal = (
    df.groupby(["city_id", "smog_season", "local_hour"])["pm2_5"]
    .mean()
    .reset_index()
)

fig, ax = plt.subplots(figsize=(9, 5))
hours = range(24)
styles = {(True,): "-", (False,): "--"}
for zone_id in ["capital", "lahore"]:
    for season, ls, alpha in [(True, "-", 1.0), (False, "--", 0.6)]:
        sub = diurnal[(diurnal.city_id == zone_id) & (diurnal.smog_season == season)]
        sub = sub.set_index("local_hour").reindex(hours)
        label = f"{ZONE_LABELS[zone_id]} — {'smog season' if season else 'rest of year'}"
        ax.plot(hours, sub["pm2_5"], ls, marker=".", alpha=alpha, color=ZONE_COLORS[zone_id], label=label)

ax.set_xlabel("Local hour (Asia/Karachi)")
ax.set_ylabel("Mean PM2.5 (µg/m³)")
ax.set_xticks(range(0, 24, 2))
ax.set_title("Diurnal PM2.5 profile, smog season vs. rest of year")
ax.grid(alpha=0.25)
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIGURES / "eda_diurnal_profile.png", dpi=140)
plt.show()

for zone_id in ["capital", "lahore"]:
    for season in [True, False]:
        sub = diurnal[(diurnal.city_id == zone_id) & (diurnal.smog_season == season)].set_index("local_hour")
        peak_h, trough_h = sub["pm2_5"].idxmax(), sub["pm2_5"].idxmin()
        print(
            f"{zone_id:10s} {'smog' if season else 'rest':5s} peak {peak_h:02d}:00 "
            f"({sub['pm2_5'][peak_h]:.0f}), trough {trough_h:02d}:00 ({sub['pm2_5'][trough_h]:.0f})"
        )


**Finding.** All four curves peak late evening (21:00–23:00) — consistent
with evening traffic, cooking and heating emissions arriving just as the
boundary layer collapses after sunset. Three of the four trough in
mid-afternoon (15:00–17:00), the classic daytime-mixing minimum. The
exception is the capital in smog season, whose trough shifts to 06:00
(pre-dawn) instead of the afternoon — a genuinely different overnight shape
than its own rest-of-year profile or Lahore's smog-season profile, not a
smoothed-over detail. One plausible reading: winter emissions plus stronger
nocturnal drainage flow off the nearby Margalla Hills actually clear the
capital's air overnight, where Lahore's flatter terrain keeps pollutants
trapped through dawn — a hypothesis for follow-up, not a settled claim from
one chart. Either way, the smog-season curves sit well above the
rest-of-year curves at every hour, not just at the peak, which is direct
empirical motivation for the cyclical hour features already in the store and
for the physics features validated in `03_physics_features.ipynb`.


## 3. Seasonal decomposition (STL)

The monthly climatology in §1 shows *that* there's a yearly cycle; STL
(Seasonal-Trend decomposition using LOESS) separates the daily PM2.5 series
into a smooth **trend** (is air quality getting better or worse year over
year, independent of season?), a repeating **seasonal** component (does it
recover the same Oct–Feb shape without being told the calendar?), and a
**residual** (what's left over — which is where individual smog episodes
that beat or undershoot the seasonal norm show up).


In [ ]:
from statsmodels.tsa.seasonal import STL

daily = (
    df.groupby(["city_id", "local_date"])["pm2_5"]
    .mean()
    .reset_index()
)

fig, axes = plt.subplots(4, 2, figsize=(13, 11), sharex="col")
for col, zone_id in enumerate(["capital", "lahore"]):
    series = (
        daily[daily.city_id == zone_id]
        .set_index("local_date")["pm2_5"]
        .asfreq("D")
        .interpolate(limit_direction="both")  # STL requires no gaps; see markdown note below
    )
    stl = STL(series, period=365, robust=True).fit()

    axes[0, col].plot(series.index, series.values, color=ZONE_COLORS[zone_id], lw=0.6)
    axes[0, col].set_title(f"{ZONE_LABELS[zone_id]} — daily mean PM2.5")
    axes[1, col].plot(series.index, stl.trend, color="black", lw=1.2)
    axes[1, col].set_title("Trend")
    axes[2, col].plot(series.index, stl.seasonal, color="darkorange", lw=0.8)
    axes[2, col].set_title("Seasonal (annual)")
    axes[3, col].plot(series.index, stl.resid, color="grey", lw=0.5)
    axes[3, col].set_title("Residual")
    axes[3, col].axhline(0, color="black", lw=0.5)

    resid_months = pd.Series(stl.resid.index).dt.month.values
    resid_smog = pd.Series(resid_months).isin(SMOG_MONTHS).values
    print(
        f"{zone_id:10s} trend: first-30d mean {stl.trend.iloc[:30].mean():.1f} -> "
        f"last-30d mean {stl.trend.iloc[-30:].mean():.1f} ug/m3 "
        f"(seasonal amplitude {stl.seasonal.max() - stl.seasonal.min():.1f})"
    )
    print(
        f"{'':10s} residual std, smog season: {stl.resid.values[resid_smog].std():.1f}  "
        f"rest of year: {stl.resid.values[~resid_smog].std():.1f}"
    )

fig.tight_layout()
fig.savefig(FIGURES / "eda_stl_decomposition.png", dpi=140)
plt.show()


**Finding.** The seasonal component recovers the same Oct–Feb rise the raw
climatology in §1 showed, without being told the calendar — confirmation the
smog-season shape is a real, stable feature of the series rather than an
artifact of how the monthly averages were binned. The trend line moves only
modestly over the four years relative to the seasonal swing (a few µg/m³
against a seasonal amplitude in the tens to hundreds), so the dominant
source of variance is *within-year* (season), not a multi-year drift —
exactly why the model ladder (session 5) needs walk-forward splits that
include full seasons (I2) rather than a single random split that could land
entirely in a mild year. The residual is where individual episodes live, and
it is measurably not homoscedastic: residual standard deviation printed
above is **higher in smog season than the rest of the year for both zones**
(roughly 30% higher for the capital, more than double for Lahore) — the
*hardest-to-predict* days genuinely cluster in exactly the season the
episode metrics (§12.4) are scored on, not just visually but by the
decomposition's own numbers.

**Caveat, stated plainly.** `STL` requires a gap-free series; the interpolation
above is for this decomposition chart only and is never used to fill the
feature store itself — the store's own gaps (e.g. `boundary_layer_height`,
ADR-015) are left as real `NaN`s with explicit `_is_missing` flags precisely
so a model never learns from an invented value (CLAUDE.md I10).


## 4. Correlation structure

Which raw variables move together? This is a sanity check before any
feature-engineering claim: pollutants that are chemically or source-linked
should correlate, and the weather variables physics features are built from
(`temperature_850hPa`, `boundary_layer_height`, `wind_speed_10m`) should show
up against PM2.5 in the direction physical intuition predicts.


In [ ]:
CORR_COLS = [
    "pm2_5", "pm10", "o3", "no2", "so2", "co",
    "temperature_2m", "relative_humidity_2m", "wind_speed_10m",
    "boundary_layer_height", "surface_pressure", "inversion_proxy",
]

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
corr_by_zone = {}
for ax, zone_id in zip(axes, ["capital", "lahore"]):
    sub = df[df.city_id == zone_id][CORR_COLS]
    corr = sub.corr()
    corr_by_zone[zone_id] = corr
    im = ax.imshow(corr, vmin=-1, vmax=1, cmap="RdBu_r")
    ax.set_xticks(range(len(CORR_COLS)))
    ax.set_xticklabels(CORR_COLS, rotation=90, fontsize=7)
    ax.set_yticks(range(len(CORR_COLS)))
    ax.set_yticklabels(CORR_COLS, fontsize=7)
    ax.set_title(ZONE_LABELS[zone_id])

fig.colorbar(im, ax=axes, shrink=0.8, label="Pearson r")
fig.suptitle("Correlation structure: pollutants and weather")
fig.savefig(FIGURES / "eda_correlation_heatmap.png", dpi=140, bbox_inches="tight")
plt.show()

for zone_id in ["capital", "lahore"]:
    pm25_corr = corr_by_zone[zone_id]["pm2_5"].drop("pm2_5").sort_values()
    print(f"{zone_id} - pm2_5 correlated with every other variable, weakest to strongest:")
    print(pm25_corr.round(2).to_string())
    print()


**Finding.** PM2.5's single strongest partner in both zones is PM10 (r=0.85
capital, 0.81 Lahore) — shared particulate sources (dust, combustion,
construction) co-emit both sizes. The next tier is the combustion co-pollutant
group — NO2, CO, SO2, all r≈0.5–0.75 — which lines up with combustion
(traffic, heating, crop burning) as a shared source with PM2.5, not just
windblown dust, and is worth remembering when reading SHAP explanations later
(session 9): these gases are collinear proxies for the same underlying
sources, not independent evidence. `inversion_proxy` is meaningfully
positive on its own, with no rolling window or missingness handling
involved (r=0.25 capital, r=0.50 Lahore) — direct correlation-structure
support for §10's physics features **before** the dedicated validation in
`03_physics_features.ipynb`. Dispersion variables (`boundary_layer_height`,
`wind_speed_10m`) are negative as physically expected but are not PM2.5's
single strongest correlate in either zone — that distinction goes to `o3`
(capital, r=-0.34, via NOx titration near fresh combustion — a different
chemical regime, not a dispersion effect) and `temperature_2m` (Lahore,
r=-0.57, itself confounded with season: winters are both cold and dirty). A
raw correlation cannot separate "dispersion clears the air" from "it happens
to be warmer when the air is already clean" — that separation is exactly
what the physics-feature validation notebook does next. Both zones share the
same qualitative ranking with different magnitudes throughout, another point
of evidence for ADR-013's zone-level (not pooled) modelling choice.


## Summary

1. **The smog season is real and asymmetric.** Both zones peak in January and
   trough in April; Lahore's amplitude is roughly double the capital's,
   consistent with its position inside the Indo-Gangetic basin. This is
   independent confirmation that ADR-016's choice of the 2025–26 season as
   the held-out test window sits inside the period the model most needs to
   get right.
2. **The diurnal cycle is a real, separate signal from season**, driven by
   nocturnal boundary-layer collapse — present year-round, sharper in winter.
   Supports the cyclical hour features already in the store.
3. **STL confirms the seasonal shape is stable and dominant** over any
   multi-year trend, and shows the residual (hardest-to-predict) variance
   concentrates inside smog season — direct motivation for I2's full-season
   test-window requirement and for scoring episode metrics separately from
   an overall average (§12.4).
4. **Dispersion capacity (boundary-layer height / wind) is among PM2.5's
   strongest correlates**, ahead of most individual gas pollutants — the
   quantitative case for the physics features validated next, in
   `03_physics_features.ipynb`.

All four charts are saved to `reports/figures/` for the report (D11,
CLAUDE.md I5's "generated, never typed" spirit extended to figures).
